# Exploratory Data Analysis

Load CSVs from `data/` (or inside the first `.tgz/.tar.gz` found), then run overview, univariate, and bivariate checks using `EDAAnalyzer`.

In [2]:
import sys
from pathlib import Path

# Ensure repo root is on path so `import gnn` works when running from notebooks/
REPO_ROOT = Path.cwd()
for parent in [Path.cwd(), *Path.cwd().parents]:
    if (parent / "src" / "gnn").exists():
        REPO_ROOT = parent
        break
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))
print(f"Repo root: {REPO_ROOT}")

Repo root: C:\Users\TZ\repo\gnn


In [3]:
import tarfile
import pandas as pd
from pathlib import Path

def find_data_dir() -> Path:
    start = Path.cwd()
    for path in [start, *start.parents]:
        candidate = path / "data"
        if candidate.exists():
            return candidate
    return start / "data"

DATA_DIR = find_data_dir()
DATA_DIR.mkdir(parents=True, exist_ok=True)
extracted_dir = DATA_DIR / "_extracted"
extracted_dir.mkdir(parents=True, exist_ok=True)

def collect_csv_paths() -> list[Path]:
    csvs = sorted(DATA_DIR.glob("*.csv"))
    csvs += sorted(extracted_dir.rglob("*.csv"))
    if csvs:
        return csvs
    archives = sorted(list(DATA_DIR.glob("*.tgz")) + list(DATA_DIR.glob("*.tar.gz")))
    if archives:
        archive = archives[0]
        with tarfile.open(archive) as tar:
            tar.extractall(extracted_dir)
        csvs = sorted(extracted_dir.rglob("*.csv"))
        if csvs:
            return csvs
    raise FileNotFoundError(f"No CSVs found in {DATA_DIR}; add sample.csv or a CSV-containing archive.")

# Load all CSVs; keyed by file stem
csv_paths = collect_csv_paths()
dataframes = {}
for path in csv_paths:
    name = path.stem
    df_loaded = pd.read_csv(path)
    dataframes[name] = df_loaded
    print(f"Loaded {path} as '{name}' with shape {df_loaded.shape}")

print(f"Total CSV files loaded: {len(dataframes)} -> {list(dataframes.keys())}")
PRIMARY_NAME = next(iter(dataframes))
df = dataframes[PRIMARY_NAME]
df.head()

Loaded C:\Users\TZ\repo\gnn\data\_extracted\aml_accounts.csv as 'aml_accounts' with shape (9914140, 3)
Loaded C:\Users\TZ\repo\gnn\data\_extracted\aml_transactions.csv as 'aml_transactions' with shape (45403506, 8)
Total CSV files loaded: 2 -> ['aml_accounts', 'aml_transactions']


,acct_id,bank_number,account_number
0,80651BC10|860D7FB40,80651BC10,860D7FB40
1,855032D90|860D7FF70,855032D90,860D7FF70
2,8538FBC70|860D78ED0,8538FBC70,860D78ED0
3,81C58CFF0|860D96510,81C58CFF0,860D96510
4,8309462C0|860D8D890,8309462C0,860D8D890


In [9]:
df['bank_number'].value_counts().head(10)

bank_number
100417440    1793
80010DE70    1756
80010EC60    1756
80010E7E0    1741
80010D200    1734
8025A3170    1733
8003AE940    1721
80010E220    1712
80010E550    1692
80010DC10    1690
Name: count, dtype: int64

In [10]:
df['account_number'].value_counts().head(10)

account_number
860D7FB40    1
860D7FF70    1
860D78ED0    1
860D96510    1
860D8D890    1
860D70900    1
860D70A90    1
860D8A7B0    1
860DFC080    1
860E242F0    1
Name: count, dtype: int64

In [4]:
# Optional: peek at other dataframes
for name, df_other in dataframes.items():
    if name == PRIMARY_NAME:
        continue
    print(f"Secondary dataframe: {name}, shape={df_other.shape}")
    display(df_other.head())

Secondary dataframe: aml_transactions, shape=(45403506, 8)


,from_account_id,to_account_id,timestamp,amount_paid,amount_received,paid_currency,received_currency,payment_type
0,80651BC10|860D7FB40,8724EDD00|873A01A20,2019-01-01 00:27:00,6.37,6.37,US Dollar,US Dollar,Credit Card
1,855032D90|860D7FF70,85BBFFA00|918B03DE0,2019-01-01 00:13:00,31.35,31.35,US Dollar,US Dollar,Credit Card
2,8538FBC70|860D78ED0,830659890|9BBE28390,2019-01-01 00:10:00,56.83,56.83,US Dollar,US Dollar,Credit Card
3,81C58CFF0|860D96510,9B068EB80|9F61C3160,2019-01-01 00:18:00,9.37,9.37,US Dollar,US Dollar,Credit Card
4,8309462C0|860D8D890,9237AAB50|A127E15C0,2019-01-01 00:29:00,2.41,2.41,US Dollar,US Dollar,Credit Card


In [7]:
from gnn.analysis import EDAAnalyzer

# TARGET = "label"  # update to your target column if available
OUTPUT_DIR = Path("artifacts/eda") / PRIMARY_NAME
analyzer = EDAAnalyzer(df)

overview_paths = analyzer.data_overview(OUTPUT_DIR)
print("Overview files:", overview_paths)

ValueError: Cannot describe a DataFrame without columns

In [ ]:
# TARGET = "label"  # update to your target column if available
OUTPUT_DIR = Path("artifacts/eda") / PRIMARY_NAME
analyzer = EDAAnalyzer(df_other)

overview_paths = analyzer.data_overview(OUTPUT_DIR)
print("Overview files:", overview_paths)

In [20]:
# focus on us ach only, clapse on date
df_sample = df_other.loc[(df_other['paid_currency']=='US Dollar') & (df_other['payment_type']=='ACH')]
df_sample['date'] = pd.to_datetime(df_sample['timestamp']).dt.date
df_sample[["bank_from","acct_from"]] = df_sample["from_account_id"].str.split("|", expand=True)
df_sample[["bank_to","acct_to"]] = df_sample["to_account_id"].str.split("|", expand=True)

df_sample = df_sample[["bank_from","acct_from","bank_to","acct_to","date","amount_paid"]]
df_sample = df_sample.rename(columns={'amount_paid':'amt'})
df_sample.head(5)

C:\Users\TZ\AppData\Local\Temp\ipykernel_24504\141180939.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sample['date'] = pd.to_datetime(df_sample['timestamp']).dt.date
C:\Users\TZ\AppData\Local\Temp\ipykernel_24504\141180939.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sample[["bank_from","acct_from"]] = df_sample["from_account_id"].str.split("|", expand=True)
C:\Users\TZ\AppData\Local\Temp\ipykernel_24504\141180939.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a 

,bank_from,acct_from,bank_to,acct_to,date,amt
57,A306732E0,B19676F40,9C7A02740,B19676D10,2019-01-31,115.43
64,A1D342960,B19664770,8604EE500,B19668520,2019-01-31,2.85
121,803A70E80,8066F9560,8025799A0,806EE7500,2019-01-20,15057.01
176,9D31C3340,B19687690,82E78C7F0,B19687370,2019-01-31,-1739.79
258,8051EF8E0,94DF22CA0,91D2F9C10,B1969F4D0,2019-01-31,2087.74


In [22]:
# Univariate analysis
num_cols = ['amt']
cat_cols = ['bank_from','bank_to','date']
continuous_subset = num_cols  # adjust as needed
categorical_subset = cat_cols  # adjust as needed

uni_paths = analyzer.univariate_analysis(
    continuous_cols=continuous_subset,
    categorical_cols=categorical_subset,
    output_dir=OUTPUT_DIR / "univariate",
    bins=10,
    winsor_limits=(0.01, 0.99),
)
print("Univariate plots:", uni_paths)

Univariate plots: {'continuous': [], 'categorical': []}


In [ ]:
# # Bivariate analysis
# if len(num_cols) >= 2:
#     x, y = num_cols[0], num_cols[1]
#     group_col = cat_cols[0] if cat_cols else None
#     bi_paths = analyzer.bivariate_analysis(x=x, y=y, group_col=group_col, output_dir=OUTPUT_DIR / "bivariate")
#     print("Bivariate outputs:", bi_paths)
# else:
#     print("Not enough numeric columns for bivariate analysis.")

In [23]:
df_sample.to_csv(r"C:\Users\TZ\repo\gnn\data\trans_sample.csv", index=False, compression='gzip')